# Neural Network From Scratch

Building a fully connected neural network from scratch using only NumPy.

## 1. Theory & Intuition

### Why Neural Networks?
- Logistic regression = single neuron with sigmoid activation
- Single neuron can only draw **linear** decision boundaries
- XOR problem: no straight line can separate diagonally opposite classes
- Multi-layer networks learn **non-linear** decision boundaries

### Key Concepts
- **Neuron:** `output = activation(W·x + b)`
- **Without activations:** stacking layers collapses to single linear transformation — proved mathematically: W2(W1x + b1) + b2 = W_combined·x + b_combined
- **Forward pass:** data flows input → hidden → output
- **Backward pass:** gradients flow output → hidden → input via chain rule
- **ReLU derivative:** 1 where active, 0 where not (dying ReLU problem)

## 2. From Scratch Implementation

In [5]:
import numpy as np

def sigmoid(z):
    return 1/(1+np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def forward_pass(X, W1, b1, W2, b2):
    z1 = np.dot(X, W1) + b1
    a1 = relu(z1)

    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)

    return a1, a2

def binary_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)  # prevent log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def backward_pass(X, y_true, a1, a2, W2):
    n = X.shape[0]
    
    # Output layer gradient
    dL_da2 = a2 - y_true          # derivative of BCE loss
    dL_dz2 = dL_da2               # sigmoid derivative absorbed
    
    # Gradients for W2 and b2
    dW2 = (1/n) * np.dot(a1.T, dL_dz2)
    db2 = (1/n) * np.sum(dL_dz2, axis=0)
    
    # Hidden layer gradient — chain rule through W2
    dL_da1 = np.dot(dL_dz2, W2.T)
    dL_dz1 = dL_da1 * (a1 > 0)   # ReLU derivative
    
    # Gradients for W1 and b1
    dW1 = (1/n) * np.dot(X.T, dL_dz1)
    db1 = (1/n) * np.sum(dL_dz1, axis=0)
    
    return dW1, db1, dW2, db2

def train(X, y, epochs=1000, lr=0.01):
    # initialize weights
    np.random.seed(42)
    W1 = np.random.randn(3, 4) * 0.01
    b1 = np.zeros(4)
    W2 = np.random.randn(4, 1) * 0.01
    b2 = np.zeros(1)
    
    for epoch in range(epochs):
        # forward pass
        a1, a2 = forward_pass(X, W1, b1, W2, b2)
        
        # compute loss
        loss = binary_cross_entropy(y, a2)
        
        # backward pass
        dW1, db1, dW2, db2 = backward_pass(X, y, a1, a2, W2)
        
        # update weights
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    return W1, b1, W2, b2


np.random.seed(42)

# 5 samples, 3 features
X = np.random.randn(5, 3)

# Hidden layer: 3 features → 4 neurons
W1 = np.random.randn(3, 4)
b1 = np.zeros(4)

# Output layer: 4 neurons → 1 output
W2 = np.random.randn(4, 1)
b2 = np.zeros(1)

a1, output = forward_pass(X, W1, b1, W2, b2)
print("Output shape:", output.shape)
print("Output values:", output)

y_true = np.array([[1], [0], [1], [0], [1]])
loss = binary_cross_entropy(y_true, output)
print(f"Loss: {loss:.4f}")

y_true = np.array([[1], [0], [1], [0], [1]])
W1, b1, W2, b2 = train(X, y_true, epochs=1000, lr=0.1)


Output shape: (5, 1)
Output values: [[0.4811108 ]
 [0.46320361]
 [0.48025389]
 [0.5764219 ]
 [0.69218078]]
Loss: 0.6628
Epoch 0: Loss = 0.6932
Epoch 100: Loss = 0.6731
Epoch 200: Loss = 0.6727
Epoch 300: Loss = 0.6696
Epoch 400: Loss = 0.6385
Epoch 500: Loss = 0.5779
Epoch 600: Loss = 0.5202
Epoch 700: Loss = 0.3640
Epoch 800: Loss = 0.2168
Epoch 900: Loss = 0.1341


## 3. PyTorch Implementation

In [ ]:
# PyTorch implementation coming in notebook 02_pytorch_basics

## 4. Train & Evaluate

In [ ]:
# Training already implemented in section 2 above
# See backward_pass and train functions

## 5. Key Takeaways

### Key Takeaways
1. A neural network is stacked neurons — each does weighted sum + activation
2. Without activation functions, any depth collapses to a single linear layer (proved mathematically)
3. ReLU derivative is 1 where active, 0 where not — neurons with z<=0 get zero gradient (dying ReLU)
4. Backprop is chain rule applied backwards through layers — same math as logistic regression gradient
5. Loss decreased from ~0.69 to ~0.13 in 1000 epochs — network successfully learned